# Train Model Notebook
This notebook illustrates a two-step modeling process: first a simple baseline model, then a more flexible Random Forest. The goal is to teach students how to compare model performance and choose improvements.


## 1. Load processed data
Load the cleaned dataset produced by the prepare notebook and inspect the available features.


In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from seattle_energy.model_training import load_dataset, get_feature_columns, TARGET_COLUMN
from sklearn.model_selection import train_test_split

df = load_dataset()
features = get_feature_columns(df)
print(f"Loaded dataset with {len(df)} rows and {len(features)} feature columns.")
print("Target column:", TARGET_COLUMN)
df.head()


ImportError: numpy._core.multiarray failed to import

In [2]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

In [4]:
%pip install -r ../requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
langchain 0.0.342 requires anyio<4.0, but you have anyio 4.13.0 which is incompatible.
langchain 0.0.342 requires numpy<2,>=1, but you have numpy 2.4.4 which is incompatible.
radcad 0.8.4 requires pandas<2.0.0,>=1.0.0, but you have pandas 2.3.3 which is incompatible.



  Using cached aiohttp_asgi_connector-1.1.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 3.2 MB/s eta 0:00:00
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
INFO: pip is looking at multiple versions of jupyter-server to determine which version is compatible with other requirements. This could take a while.
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyter_events-0.12.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached 

## 2. Baseline model: linear regression on log surface
A baseline model helps students understand whether the advanced model is actually better. We use a simple linear regression on the logarithm of building surface area.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

baseline_feature = ["log_surface"]
if not set(baseline_feature).issubset(df.columns):
    raise ValueError("Required baseline feature not found in dataset")

X = df[baseline_feature]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
y_pred_train = baseline_model.predict(X_train)
y_pred_test = baseline_model.predict(X_test)

def evaluate(y_true, y_pred, name=""):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name}: R2={r2:.4f}, MAE={mae:.2f}, RMSE={rmse:.2f}")

evaluate(y_train, y_pred_train, "Baseline Train")
evaluate(y_test, y_pred_test, "Baseline Test")


## 3. Random Forest with the full feature set
Now use the full set of engineered features and a more flexible model. This section follows the existing `model_training.py` logic while making the training process visible in the notebook.


In [ ]:
from seattle_energy.model_training import build_search_model, save_bento_model

full_features = [col for col in features if col not in ["OSEBuildingID", TARGET_COLUMN]]
X = df[full_features]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

search = build_search_model()
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best hyperparameters:", search.best_params_)

evaluate(y_train, best_model.predict(X_train), "Random Forest Train")
evaluate(y_test, best_model.predict(X_test), "Random Forest Test")


## 4. Save the model for serving
The final step exports the best trained model to the BentoML store, ready for deployment.


In [ ]:
save_bento_model(best_model, full_features)
print("Saved BentoML model as random_forest_energy:latest")


## 5. Residuals and prediction visualization
Examine the residuals and compare predicted vs. actual values to assess model fit quality.

In [ ]:
# Residuals = actual - predicted
residuals_train = y_train - best_model.predict(X_train)
residuals_test = y_test - best_model.predict(X_test)

# Plot residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs. predicted values (train)
axes[0].scatter(best_model.predict(X_train), residuals_train, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Energy (kBtu)')
axes[0].set_ylabel('Residuals (kBtu)')
axes[0].set_title('Train: Residuals vs. Predictions')
axes[0].grid(True, alpha=0.3)

# Residuals vs. predicted values (test)
axes[1].scatter(best_model.predict(X_test), residuals_test, alpha=0.5, color='orange')
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted Energy (kBtu)')
axes[1].set_ylabel('Residuals (kBtu)')
axes[1].set_title('Test: Residuals vs. Predictions')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Residuals summary:")
print(f"Train residuals: mean={residuals_train.mean():.2f}, std={residuals_train.std():.2f}")
print(f"Test residuals: mean={residuals_test.mean():.2f}, std={residuals_test.std():.2f}")

## 6. Predicted vs. Actual Energy Consumption
Visualize how well the model predictions align with actual energy values across the test set.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot: actual vs predicted on test set
y_pred_test_rf = best_model.predict(X_test)
ax.scatter(y_test, y_pred_test_rf, alpha=0.5, edgecolors='k', linewidth=0.5)

# Perfect prediction line
min_val = min(y_test.min(), y_pred_test_rf.min())
max_val = max(y_test.max(), y_pred_test_rf.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')

ax.set_xlabel('Actual Energy (kBtu)', fontsize=12)
ax.set_ylabel('Predicted Energy (kBtu)', fontsize=12)
ax.set_title('Test Set: Predicted vs. Actual Energy Consumption', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature importance analysis (MDI)
Model-based Importance (MDI) shows which features have the highest impact on predictions according to the Random Forest model.

In [ ]:
# Extract feature importances from the Random Forest
feature_importances = best_model.feature_importances_
feature_names = full_features

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print("Top 10 most important features:")
print(importance_df.head(10))

# Plot top 15 features
fig, ax = plt.subplots(figsize=(10, 6))
top_features = importance_df.head(15)
ax.barh(range(len(top_features)), top_features['importance'].values)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Top 15 Most Important Features (MDI)', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Permutation importance
A model-agnostic method to measure feature importance by calculating the drop in performance when a feature's values are randomly shuffled.

In [ ]:
from sklearn.inspection import permutation_importance

# Compute permutation importance on the test set
perm_importance = permutation_importance(
    best_model, X_test, y_test, 
    n_repeats=10, random_state=42, n_jobs=-1
)

# Create a DataFrame
perm_importance_df = pd.DataFrame({
    'feature': full_features,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("Top 10 features by permutation importance:")
print(perm_importance_df.head(10))

# Plot top 15 features
fig, ax = plt.subplots(figsize=(10, 6))
top_perm_features = perm_importance_df.head(15)
ax.barh(range(len(top_perm_features)), top_perm_features['importance_mean'].values,
        xerr=top_perm_features['importance_std'].values)
ax.set_yticks(range(len(top_perm_features)))
ax.set_yticklabels(top_perm_features['feature'].values)
ax.set_xlabel('Permutation Importance', fontsize=12)
ax.set_title('Top 15 Most Important Features (Permutation)', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()